# 11 — configuring the network

The ten notebooks before this one built one model for one experiment. This one
is about **saying which experiment you have**, because until now you could not:
the list of histograms was a literal in the source, and the physics each
histogram carries was a dictionary of six cases written out by hand.

A time-resolved FRET measurement is four lists and nothing else.

| | |
|---|---|
| **samples** | what was in the cuvette. A sample carries a donor, an acceptor, or both. |
| **excitations** | the laser pulses, and *how strongly* each reaches each chromophore. |
| **detectors** | each sees one colour and has its own instrument response and timing shift. |
| **channels** | the histograms somebody actually recorded: a sample, a pulse, a detector, a polarisation. |

Everything else is derived — which physics each histogram carries, which
unknowns exist, which of them the data can move, and what the factor graph
looks like. This notebook does no fitting. It is about the description.

Notebooks 12 to 15 each take one description and fit it.


In [1]:
import sys, json
from pathlib import Path
import numpy as np
import experiment as X


## The description, written out

Here is an MFD-PIE measurement, field by field. Nothing in this cell is a
choice about *analysis*; every line is a fact about the optical table.


In [2]:
green = X.Excitation('green', {'donor': 1.0, 'acceptor': 'EX_AG'}, delay_ns=0.0)
red   = X.Excitation('red',   {'acceptor': 1.0, 'donor': 'EX_DR'}, delay_ns=25.0)

detectors = [X.Detector('gv', 'green'), X.Detector('gh', 'green'),
             X.Detector('rv', 'red'),   X.Detector('rh', 'red')]

samples = [X.Sample('D0', ('donor',)),
           X.Sample('DA', ('donor', 'acceptor')),
           X.Sample('A0', ('acceptor',))]

pol = {'gv': 'parallel', 'gh': 'perpendicular', 'rv': 'parallel', 'rh': 'perpendicular'}
channels = [X.Channel(s.name, e, d.name, pol[d.name])
            for s in samples for d in detectors for e in ('green', 'red')]

pie = X.Experiment(name='MFD-PIE, written out here',
                   samples=samples, excitations=[green, red],
                   detectors=detectors, channels=channels,
                   interleaved=True).validate()
print(pie.summary())


MFD-PIE, written out here: 24 histograms
  samples     D0 (donor), DA (donor+acceptor), A0 (acceptor)
  excitations green at 0 ns, reaches donor x 1.0, acceptor x EX_AG, red at 25 ns, reaches acceptor x 1.0, donor x EX_DR
  detectors   gv (green), gh (green), rv (red), rh (red)
  histograms
    D0  gv_vv    parallel      donor
    D0  g2v_vv   parallel      donor (EX_DR)
    D0  gh_vh    perpendicular donor
    D0  g2h_vh   perpendicular donor (EX_DR)
    D0  rv_vv    parallel      donor
    D0  r2v_vv   parallel      donor (EX_DR)
    D0  rh_vh    perpendicular donor
    D0  r2h_vh   perpendicular donor (EX_DR)
    DA  gv_vv    parallel      donor + FRET + sensitised acceptor + acceptor directly (EX_AG)
    DA  g2v_vv   parallel      donor (EX_DR) + FRET + acceptor directly
    DA  gh_vh    perpendicular donor + FRET + sensitised acceptor + acceptor directly (EX_AG)
    DA  g2h_vh   perpendicular donor (EX_DR) + FRET + acceptor directly
    DA  rv_vv    parallel      donor + FRET + se

### The strengths are the point

A green pulse and a red pulse both reach both chromophores. What separates them
is *how strongly*: the green pulse was chosen for the donor and reaches it at
strength 1, and reaches the acceptor only through a weak path whose strength is
unknown and has to be fitted — which is why it is written as the **name of a
parameter**, `EX_AG`, rather than as a number.

Saying it this way makes direct excitation a term in the model instead of an
assumption. It is also load-bearing: the first version of this file described a
pulse by *which* chromophores it reaches, as a set, and the gate below caught it
— with that description the six cases of interleaved excitation collapse into
three, because both pulses reach both chromophores.

### Interleaved is not the same as two lasers

Two pulses in one laser period share a histogram, and the model has to separate
them; two lasers in two separate acquisitions do not. That is a different
experiment, so `interleaved` is a field of the description rather than a count
of pulses. `separate_measurements()` below has two excitations and is not
interleaved.


## The rule that replaces the table

One histogram carries: a directly excited donor, at the strength with which its
pulse reaches the donor **of those molecules**; energy transfer, if the sample
carries both chromophores and the pulse reaches the donor; a sensitised
acceptor, by the same condition; and a directly excited acceptor at the
strength with which the pulse reaches it.

Six lines of code. Here they are applied to every histogram of the description
above.


In [3]:
import inspect
print(inspect.getsource(X.scope_of))


def scope_of(sample: Sample, excitation: Excitation) -> Scope:
    """THE RULE.  Six lines, in place of a table of six cases written out by
    hand -- and it produces the cases of geometries that were never tabulated."""
    has_d = 'donor' in sample.carries
    has_a = 'acceptor' in sample.carries
    to_d = excitation.strength('donor') if has_d else 0.0
    to_a = excitation.strength('acceptor') if has_a else 0.0
    transfer = bool(has_d and has_a and to_d)
    #: the acceptor sensitised by a donor that was itself only weakly excited
    #: is second order and the model drops it; that is a modelling choice and
    #: it belongs here, where it can be read
    return Scope(donor=to_d, acceptor=to_a, fret=transfer,
                 sensitised=bool(transfer and to_d == 1.0))



In [4]:
for k, s in pie.scope_table().items():
    print(f'{k[0]:<3} {k[1]:<8} {s.describe()}')


D0  gv_vv    donor
D0  g2v_vv   donor (EX_DR)
D0  gh_vh    donor
D0  g2h_vh   donor (EX_DR)
D0  rv_vv    donor
D0  r2v_vv   donor (EX_DR)
D0  rh_vh    donor
D0  r2h_vh   donor (EX_DR)
DA  gv_vv    donor + FRET + sensitised acceptor + acceptor directly (EX_AG)
DA  g2v_vv   donor (EX_DR) + FRET + acceptor directly
DA  gh_vh    donor + FRET + sensitised acceptor + acceptor directly (EX_AG)
DA  g2h_vh   donor (EX_DR) + FRET + acceptor directly
DA  rv_vv    donor + FRET + sensitised acceptor + acceptor directly (EX_AG)
DA  r2v_vv   donor (EX_DR) + FRET + acceptor directly
DA  rh_vh    donor + FRET + sensitised acceptor + acceptor directly (EX_AG)
DA  r2h_vh   donor (EX_DR) + FRET + acceptor directly
A0  gv_vv    acceptor directly (EX_AG)
A0  g2v_vv   acceptor directly
A0  gh_vh    acceptor directly (EX_AG)
A0  g2h_vh   acceptor directly
A0  rv_vv    acceptor directly (EX_AG)
A0  r2v_vv   acceptor directly
A0  rh_vh    acceptor directly (EX_AG)
A0  r2h_vh   acceptor directly


Six distinct cases came out, and they are the six the prototype has written into
it by hand. **The gate asserts that**, key by key, without the rule having been
shown the table: two histograms land in the same derived case if and only if the
prototype gives them the same name. It also checks the derived channel lists
against the three literals in the prototype, and the JSON round trip.


In [5]:
X.check_against_prototype();


  mfd_pie(full=True)         {'channels': 24, 'scopes': 6}
  mfd_pie(full=False)        {'channels': 12}
  separate_measurements      {'channels': 8, 'scopes': 3}
  mfd_donor_excitation       {'channels': 4, 'scopes': 1}
  magic_angle_minimal        {'channels': 2, 'scopes': 1}
  json                       {'round_trip': 'exact for every geometry'}
PASS: the derived channels and scopes reproduce the hand-written ones


## The same description as JSON

`to_json` and `from_json`, and the round trip is exact — the fields and the
derived channel keys both.


In [6]:
text = pie.to_json()
print(text[:900] + '\n...')
back = X.Experiment.from_json(text)
assert back.to_dict() == pie.to_dict()
assert back.channel_keys() == pie.channel_keys()
print('\nround trip exact:', len(pie.channels), 'histograms')


{
  "name": "MFD-PIE, written out here",
  "notes": "",
  "interleaved": true,
  "samples": [
    {
      "name": "D0",
      "carries": [
        "donor"
      ]
    },
    {
      "name": "DA",
      "carries": [
        "donor",
        "acceptor"
      ]
    },
    {
      "name": "A0",
      "carries": [
        "acceptor"
      ]
    }
  ],
  "excitations": [
    {
      "name": "green",
      "excites": {
        "donor": 1.0,
        "acceptor": "EX_AG"
      },
      "delay_ns": 0.0
    },
    {
      "name": "red",
      "excites": {
        "acceptor": 1.0,
        "donor": "EX_DR"
      },
      "delay_ns": 25.0
    }
  ],
  "detectors": [
    {
      "name": "gv",
      "colour": "green"
    },
    {
      "name": "gh",
      "colour": "green"
    },
    {
      "name": "rv",
      "colour": "red"
    },
    {
      "name": "rh",
      "colour": "red"
    }
  ],
  "channels"
...

round trip exact: 24 histograms


In [7]:
Path('my_experiment.json').write_text(text)
same = X.Experiment.from_json('my_experiment.json')
print(same.name, '-', len(same.channels), 'histograms read back from the file')


MFD-PIE, written out here - 24 histograms read back from the file


## What a description refuses

A validator that never refuses anything is not a validator. Each of these is a
description that cannot be true, and each is refused with a message that names
the offending item.


In [8]:
X.check_refusals();


  refused: a channel naming a detector that does not exist
           "no detector named 'nope'; have ['g']"
  refused: two detectors with the same name
           duplicate detector name(s) ['g']
  refused: an excitation that excites nothing
           excitation 'dark' excites nothing; leave it out instead
  refused: a histogram that carries nothing
           the histogram A0/green/g carries nothing: sample 'A0' has ('acceptor',) and pulse 'green' reaches {'donor': 1.0}
  refused: three interleaved pulses
           3 interleaved pulses: the model addresses a pulse by giving each detector a second alias, so it handles exactly two. Three would need a third alias and a third window, which is not implemented.


## What the description implies

Which unknowns exist is not a separate decision. A scatter fraction and a
background fraction belong to each **histogram**; an instrument response and a
timing shift to each **detector** that was used; one acquisition scale to each
**sample**. The distance distribution, the photophysics and the calibration are
shared by all of them.


In [9]:
small = X.separate_measurements()
model = X.build(small, knots=25)
g = X.graph(small, model)
print(small.name, '-', len(model['keys']), 'histograms,', g.dim, 'free coordinates')
groups = {}
for v in g.free:
    groups.setdefault(v.group, []).append(f'{v.name}[{v.size}]' if v.size > 1 else v.name)
for grp, names in groups.items():
    print(f'  {grp:<5} {", ".join(names)}')


  maps loaded from /Users/tpeulen/dev/ucfret/investigation/pinn_pR_anisotropy/ckpt/homog/s87_env_128_v2.pt
separate donor, FRET and acceptor measurements - 8 histograms, 109 free coordinates
  hyper log10_lam
  phys  c[24], x_d0, spec_eps[33], w_a[3], w_rho[8], w_rho_a[4], r0_d, r0_a
  cal   g, l1, l2, C_GD, C_GA, C_RD, C_RA, G_GREEN, G_RED, QY_D, QY_A, EX_AG
  inst  irf_shift_g, irf_shift_r, log_scale_D0, log_scale_DA, log_scale_A0, scat_D0_g_vv, bkg_D0_g_vv, scat_D0_g_vh, bkg_D0_g_vh, scat_DA_g_vv, bkg_DA_g_vv, scat_DA_g_vh, bkg_DA_g_vh, scat_DA_r_vv, bkg_DA_r_vv, scat_DA_r_vh, bkg_DA_r_vh, scat_A0_r_vv, bkg_A0_r_vv, scat_A0_r_vh, bkg_A0_r_vh


## What the experiment can and cannot determine

The useful question about a description is not what it contains but **what it
leaves undetermined**. `identifiability` moves each coordinate and measures how
much the expected counts of each histogram move: Fisher's information, in the
coordinate the fit works in. Comparing it with the prior's own precision gives
the fraction of what the posterior will say that comes from the data.

A share near zero is conclusive — nothing in the data moves when the coordinate
does, so whatever a fit prints for it is the prior. A share near one is *not*
conclusive, because this is the diagonal: a coordinate can still be undetermined
through a correlation the diagonal cannot see.


In [10]:
mini = X.magic_angle_minimal()
mm = X.build(mini, knots=25)
rows = X.identifiability(mini, mm)
print(f'{mini.name}: {len(mm["keys"])} histograms\n')
print(f'{"unknown":<16}{"group":<7}{"data share":>11}   carried by')
for r in sorted(rows, key=lambda r: r['data_share'])[:14]:
    print(f'{r["variable"]:<16}{r["role"]:<7}{r["data_share"]:>11.4f}   {r["carried_by"]}')


  maps loaded from /Users/tpeulen/dev/ucfret/investigation/pinn_pR_anisotropy/ckpt/homog/s87_env_128_v2.pt


magic angle, labelled sample only: 2 histograms

unknown         group   data share   carried by
log10_lam       hyper       0.0000   nothing
w_rho           phys        0.0000   nothing
w_rho_a         phys        0.0000   nothing
r0_d            phys        0.0000   nothing
r0_a            phys        0.0000   nothing
g               cal         0.0000   nothing
l1              cal         0.0000   nothing
l2              cal         0.0000   nothing
C_GA            cal         0.9596   DA g_ma
C_RA            cal         0.9924   DA r_ma
C_GD            cal         0.9928   DA g_ma
C_RD            cal         0.9946   DA r_ma
c               phys        0.9947   DA r_ma, DA g_ma
EX_AG           cal         0.9963   DA r_ma


Seven of those are the anisotropy: the two fundamental anisotropies, the two
rotational-correlation-time distributions, the `g` factor and the two
polarisation-mixing constants. **At the magic angle they are not measured** —
the polarisation term is multiplied by zero — so a fit will return the prior for
all seven, and the table says so before the fit is run rather than after.

`log10_lam` comes out unidentified in every geometry, which is also right: it is
the roughness weight, and notebook 3 is about why it is integrated out rather
than fitted.


## The factor graph

The same description gives the model in the graphical sense: which unknowns
exist, and which factors touch which of them. The scopes are **measured**, by
the same perturbation — a hand-written scope list is a guess, and the
elimination order, the treewidth and the separators are all statements about
which variables share a factor, so a scope wrong by one variable is a
decomposition wrong by more.


In [11]:
fg, scopes = X.structural_graph(mini, mm)
print('variables', fg.get_number_of_variables(), ' factors', fg.get_number_of_factors())
for k, sc in scopes.items():
    print(f'\n{k[0]} {k[1]}  touches {len(sc)} unknowns')
    print('   ', ', '.join(sc))


variables 28  factors 30

DA g_ma  touches 12 unknowns
    C_GA, C_GD, G_GREEN, QY_A, QY_D, bkg_DA_g_ma, c, irf_shift_g, log_scale_DA, scat_DA_g_ma, spec_eps, x_d0

DA r_ma  touches 14 unknowns
    C_RA, C_RD, EX_AG, G_RED, QY_A, QY_D, bkg_DA_r_ma, c, irf_shift_r, log_scale_DA, scat_DA_r_ma, spec_eps, w_a, x_d0


The green histogram does not touch the red detector's crosstalk, nor the
acceptor's direct excitation; the red one does. Neither touches the anisotropy.
Nobody wrote that down — it was measured.

## Where to go next

| notebook | the experiment |
|---|---|
| `12_mfd_donor_excitation.ipynb` | one laser, four detectors, the labelled sample alone |
| `13_mfd_pie.ipynb` | two lasers interleaved, three samples, twelve histograms |
| `14_separate_measurements.ipynb` | three cuvettes, measured on their own |
| `15_magic_angle_minimal.ipynb` | two histograms, no polarisation |

Each builds its description, prints what it can and cannot determine, simulates
data from it, fits, and reports Rule 0.
